In [38]:
# from comet_ml import Experiment
from trainer import Trainer
from diffusion import Diffusion
# from unet import Unet
from diffusers.models.unet_2d_condition import UNet2DConditionModel
from diffusers import DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel
import argparse
import wandb
import torch

In [2]:
device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [33]:
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

In [25]:
model = UNet2DConditionModel.from_pretrained(
        pretrained_model_name_or_path, subfolder="unet", torch_dtype=torch.float16
    ).to(device)

diffusion = Diffusion(
    model,
    image_size=32,
    channels=32,
    timesteps=50,  # number of steps
).to(device)


c:\Users\aravi\miniconda3\envs\idl-project\lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aravi\.cache\huggingface\hub\models--runwayml--stable-diffusion-v1-5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [30]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = total_params - trainable_params

    print(f"Total Parameters: **{total_params:,}**")
    print(f"Trainable Parameters: **{trainable_params:,}**")
    print(f"Non-trainable Parameters: **{non_trainable_params:,}**")

# Example usage:
# model = YourModel()
# count_parameters(model)


In [34]:
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")

In [29]:
count_parameters(model)

Total Parameters: **859,520,964**
Trainable Parameters: **859,520,964**
Non-trainable Parameters: **0**


In [20]:
noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_name_or_path, subfolder="scheduler")
noise_scheduler.config

FrozenDict([('num_train_timesteps', 1000),
            ('beta_start', 0.00085),
            ('beta_end', 0.012),
            ('beta_schedule', 'scaled_linear'),
            ('trained_betas', None),
            ('variance_type', 'fixed_small'),
            ('clip_sample', False),
            ('prediction_type', 'epsilon'),
            ('thresholding', False),
            ('dynamic_thresholding_ratio', 0.995),
            ('clip_sample_range', 1.0),
            ('sample_max_value', 1.0),
            ('timestep_spacing', 'leading'),
            ('steps_offset', 1),
            ('rescale_betas_zero_snr', False),
            ('_use_default_values',
             ['timestep_spacing',
              'dynamic_thresholding_ratio',
              'sample_max_value',
              'rescale_betas_zero_snr',
              'prediction_type',
              'clip_sample_range',
              'variance_type',
              'thresholding']),
            ('_class_name', 'DDPMScheduler'),
            ('_diff

In [ ]:
# accelerate launch diffusers/examples/text_to_image/train_text_to_image_lora_genai.py \
#     --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
#     --dataset_name="polinaeterna/pokemon-blip-captions" \
#     --dataloader_num_workers=0 \
#     --resolution=512 \
#     --center_crop \
#     --random_flip \
#     --train_batch_size=1 \
#     --gradient_accumulation_steps=4 \
#     --max_train_steps=15000 \
#     --learning_rate=1e-04 \
#     --max_grad_norm=1 \
#     --lr_scheduler="cosine" \
#     --lr_warmup_steps=0 \
#     --output_dir="output/pokemon-lora" \
#     --report_to=wandb \
#     --checkpointing_steps=1000 \
#     --validation_prompt="A pokemon" \
#     --seed=1337

In [36]:
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

# 2. Forward embeddings and negative embeddings through text encoder
prompt = 25 * "a photo of an astronaut riding a horse on mars"
max_length = pipe.tokenizer.model_max_length
max_length

Loading pipeline components...: 100%|██████████| 7/7 [00:05<00:00,  1.23it/s]


77

In [65]:
from compel import Compel, ReturnedEmbeddingsType
from diffusers import DiffusionPipeline
import torch


class LLM:
    def __init__(self):
        self.tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")
        self.text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path, subfolder="text_encoder")

    def encode_batch_1(self, texts):
        """
        Encodes a batch of texts using the BertTokenizer.

        Args:
            texts (list of str): List of input texts to encode.

        Returns:
            torch.Tensor: Tensor of token ids for the encoded texts.
        """
        text_encoding = []
        for text in texts:
            text_tokens = self.tokenizer(
                text, return_tensors="pt"
            ).input_ids
            # print(f"text_tokens.shape = {text_tokens.shape}")
            encoder_hidden_states = self.text_encoder(
                    text_tokens, return_dict=False)[0]
            # print(f"encoder_hidden_states.shape = {encoder_hidden_states.shape}")
            text_encoding.append(encoder_hidden_states)
        ret = torch.cat(text_encoding, dim=1)
        # print(f"ret.shape = {ret.shape}")
        return ret

In [56]:
from compel import Compel
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path, subfolder="text_encoder")
compel = Compel(tokenizer=tokenizer, text_encoder=text_encoder)
embeds = compel('("a cat playing in the forest", "an impressionist oil painting").and()')

In [45]:
(embeds[0, 73, :] != embeds[0, 140, :]).any()

tensor(True)

In [49]:
embeds[0, 73, :10], embeds[0, 152, :10]

(tensor([-1.4330, -0.6558,  0.3928, -0.3701, -4.1685,  0.3087, -0.7238, -1.1895,
          1.0579,  0.9467]),
 tensor([-0.6613, -0.7404,  1.2553,  0.2065, -3.3953,  0.5141, -0.4178,  0.5303,
         -1.4020,  2.1452]))

In [66]:
llm = LLM()
llm.encode_batch_1(["green cat", "blue dog with a tail", "yellow baffoon"])

text_tokens.shape = torch.Size([1, 4])
encoder_hidden_states.shape = torch.Size([1, 4, 768])
text_tokens.shape = torch.Size([1, 7])
encoder_hidden_states.shape = torch.Size([1, 7, 768])
text_tokens.shape = torch.Size([1, 6])
encoder_hidden_states.shape = torch.Size([1, 6, 768])
ret.shape = torch.Size([1, 17, 768])


tensor([[[-0.3884,  0.0229, -0.0522,  ..., -0.4899, -0.3066,  0.0675],
         [-0.6045, -2.1100,  0.6977,  ..., -0.3181, -0.8722,  0.5462],
         [-1.8845, -0.2767,  0.8697,  ..., -1.5928, -0.7341, -1.2417],
         ...,
         [-0.3256,  1.1171,  0.5600,  ..., -0.5779,  2.3403,  0.2922],
         [-0.7212,  1.3397,  0.0551,  ..., -1.7938,  2.4515,  0.6443],
         [ 0.5712,  0.2307, -0.9092,  ..., -0.7626,  1.5487,  0.0640]]],
       grad_fn=<CatBackward0>)